In [11]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional, Literal, List
import json
import os

In [12]:
load_dotenv()

True

In [13]:
class VehicleExtraction(BaseModel):
    # TECHNIQUE 1: Chain of Thought. The LLM must reason first.
    reasoning: str = Field(
        description="Briefly explain your logic for determining the mileage, condition, battery status, and accidents based on the provided text."
    )

    # --- Category 1: Brand & Model ---
    brand: str = Field(description="The manufacturer (e.g., 'VinFast', 'Wuling', 'BYD').")
    car_model: str = Field(description="The clean car model without specs/range (e.g., 'VF3', 'VF8', 'Bingo').")

    # --- Category 2: Imputation ---
    imputed_year: Optional[int] = Field(description="The production year. Null if unknown.")
    imputed_mileage_km: Optional[int] = Field(description="Strict integer for kilometers driven.")
    imputed_condition: Optional[Literal["Mới 100%", "Đã qua sử dụng"]] = Field(description="New vs Used.")

    # --- Category 3: Feature Engineering (TECHNIQUE 2: Bounded Booleans) ---
    battery_status: Optional[Literal["Mua pin", "Thuê pin"]] = Field(description="Battery ownership.")
    is_accident_free: Optional[bool] = Field(description="True if no accidents/water damage. False if damaged.")
    has_aftermarket_mods: bool = Field(
        description="True if the seller explicitly mentions adding aftermarket parts/toys ('độ', 'lên đồ chơi', 'dán phim', 'màn hình', 'cam 360'). False if it only lists standard factory specs."
    )

In [14]:
SYSTEM_PROMPT = """
You are an expert Data Engineer specializing in the Vietnamese Electric Vehicle (EV) market.
Your task is to precisely extract and impute structured data from raw, messy car listings.

CRITICAL EXTRACTION RULES:
1. Mileage ('imputed_mileage_km'):
   - '1 vạn' or '1v' = 10,000 km. Example: '3,5 vạn' = 35000. '6.800 km' = 6800.
2. Condition ('imputed_condition'):
   - 'Mới 100%': If it mentions 'chưa lăn bánh', 'xuất hóa đơn', 'nhận xe ngay', 'đại lý', or 0km.
   - 'Đã qua sử dụng': If ODO > 500km, or mentions 'lướt', 'chính chủ', 'đã lên đồ'.
3. Battery Status ('battery_status'):
   - 'Mua pin': 'mua pin', 'đã gồm pin', 'mua đứt', 'kèm pin'.
   - 'Thuê pin': 'thuê pin', 'tiên phong', 'gói thuê'.
4. Accidents ('is_accident_free'):
   - True: 'zin keng', 'không đâm đụng', 'nguyên bản', 'keo chỉ zin', 'check hãng'.
   - False: 'va quệt', 'móp', 'lỗi'.
5. Mods ('has_aftermarket_mods'):
   - True ONLY IF the user added things (e.g., 'đã lắp 3 thanh giằng', 'lên đồ', 'dán phim', 'lên mâm').
   - False if the text just lists standard factory features (e.g., 'ABS', 'EBD', 'Pin LFP', 'Màn hình kép').
"""

In [15]:
TEST_SAMPLES = [
    {
        "id": "Thực tế 1 - Thông số kỹ thuật (Otodien)",
        "name": "BINGO",
        "description": "Wuling Bingo phiên bản 333 km là dòng xe hatchback điện 5 cửa phổ biến, nổi bật với pin LFP 31,9 kWh, phạm vi di chuyển 333 km (CLTC). Xe trang bị mô-tơ điện công suất 50 kW (67 mã lực), mô-men xoắn 150 Nm, tốc độ tối đa 100-120 km/h, kích thước nhỏ gọn 3.950 x 1.708 x 1.580 mm, phù hợp nhu cầu di chuyển đô thị. -Thông số chi tiết Wuling Bingo 333 km:-Động cơ & Vận hành:-Công suất: 50 kW (67 mã lực).-Mô-men xoắn: 150 Nm.-Tốc độ tối đa: 100 - 120 km/h.-Hệ dẫn động: Cầu trước.-Pin & Sạc:-Dung lượng pin: 31,9 kWh (Pin LFP).-Phạm vi di chuyển: 333 km (theo chu trình CLTC).-Thời gian sạc AC (3,3 kW): Khoảng 9,5 giờ hoặc 8 giờ (20-100%).-Sạc nhanh (DC): Có hỗ trợ, mất khoảng 4 giờ (20-100%).-Kích thước & Trọng lượng:-Dài x Rộng x Cao: 3.950 x 1.708 x 1.580 mm.-Chiều dài cơ sở: 2.560 mm.-Khoảng sáng gầm: 160 mm.-Lốp: 185/60R15.-Tiện nghi & An toàn:-Nội thất: Màn hình kép 10,25 inch (bảng đồng hồ và giải trí), ghế da.-An toàn: Phanh đĩa, phanh tay điện tử, camera lùi, túi khí phía trước, ABS, EBD."
    },
    {
        "id": "Thực tế 2 - Xe cũ ODO thấp (Otodien)",
        "name": "VinFast VF3 2024",
        "description": "Khai xuân năm mới. Giá nào cũng bán-Xe đẹp zin keng-Lăn bánh 6.800 km-Chính chủ-Đã lắp 3 thanh giằng ngang, ghế da, cốp trước …-Xe hoạt động ổn định. Đi chắc chắn-Giá năm mới 1xx tr"
    },
    {
        "id": "Thực tế 3 - Quảng cáo xe mới (Otodien)",
        "name": "🚗 VINFAST VF3 – GIÁ CHỈ 284 TRIỆU (pin – vay 85%)",
        "description": "🚗 VINFAST VF3 – GIÁ CHỈ 284 TRIỆU (đã gồm pin – vay 85%)-💰 Trả trước nhẹ tênh, nhận xe ngay!-ƯU ĐÃI CỰC SÂU ĐẾN 31/02/2026-🎁 ƯU ĐÃI ĐẶC BIỆT CHO KHÁCH HÀNG-🔥 Giảm 6% giá xuất hóa đơn-🔥 Tặng 1 năm bảo hiểm -🔥 Giảm thêm 5% cho khách hàng là công an và quân đội-Khách hàng mua xe trong thời gian ưu đãi nhận thêm:-📞 Liên hệ tư vấn – lái thử & nhận báo giá tốt nhất-Hiếu VinFast – ***"
    }
]

In [16]:
def test_openai(prompt: str) -> str:
    try:
        client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            response_format=VehicleExtraction,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"OpenAI Error: {e}"

In [17]:
def test_gemini(prompt: str) -> str:
    """Modified to use the correct Gemini 2.0 Flash SDK implementation."""
    try:
        client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
        response = client.models.generate_content(
            model='gemini-3-flash-preview',
            contents=f"{SYSTEM_PROMPT}\n\nExtract from this listing:\n{prompt}",
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=VehicleExtraction,
            ),
        )
        return response.text
    except Exception as e:
        return f"Gemini Error: {e}"

In [18]:
for sample in TEST_SAMPLES:
        print(f"\nTesting: {sample['id']}")
        prompt_text = f"Tên xe: {sample['name']}\nMô tả: {sample['description']}"

        print("\nOpenAI Result:")
        print(test_openai(prompt_text))

        print("\nGemini Result:")
        print(test_gemini(prompt_text))


Testing: Thực tế 1 - Thông số kỹ thuật (Otodien)

OpenAI Result:
{"reasoning":"The listing does not provide specific mileage information, but since it mentions the model is new with features highlighting efficiency and modern technology, I will impute the condition as 'Mới 100%' since it does not mention any prior usage or existing mileage. For battery status, there are no terms indicating rental; thus, it is categorized as 'Mua pin'. There is no mention of accidents or modifications, so I can mark them as accident-free and no aftermarket modifications.","brand":"Wuling","car_model":"Bingo","imputed_year":null,"imputed_mileage_km":null,"imputed_condition":"Mới 100%","battery_status":"Mua pin","is_accident_free":true,"has_aftermarket_mods":false}

Gemini Result:
{"reasoning":"The text is a technical specification summary for the Wuling Bingo 333 km model. It provides standard factory features like a 10.25-inch screen, LFP battery, and safety systems (ABS, EBD), but lacks specific detai